# Day 5 (Tue Aug 18) — Let's build GPT (1h56m) — the main event

He types EVERYTHING live from an empty notebook — so this starter is just the data + targets. Work in tandem: pause when he names a thing, build it, unpause to compare. This notebook = dev scratchpad (his gpt-dev); consolidate into gpt.py at the end (CS336 tests import a .py).

**arc:** read data → encode/decode → batches → bigram baseline (val ~2.5) → the mathematical trick → single head → multi-head → feedforward → blocks + residuals + layernorm → scale up
**targets:** bigram baseline val ~2.5 · final GPT (n_embd 384, 6 layers, 6 heads, block 256, ~10M params) **val ~1.48** on shakespeare chars
**reuse from my week:** embeddings, Linear, cross-entropy, training loop, lr decay, eval-mode discipline, param-count alarm, shape walks — all of it appears again here

In [740]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

In [741]:
BATCH_SIZE = 4 # how many independent sequences will we process in parallel?
BLOCK_SIZE = 8 # what is the maximum context length for predictions?
TRAINING_STEPS = 10000 
N_EMBEDDINGS = 32

In [742]:
# input.txt already downloaded (tiny shakespeare, 1,115,394 chars) — no wget needed
with open('input.txt', 'r') as f:
    text = f.read()
print(len(text))
print(text[:200])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [743]:
unique_text = sorted(set(text))
vocab_size = len(unique_text)
itos= {ch: i for ch, i in enumerate(unique_text)}
stoi= {i: ch for ch, i in enumerate(unique_text)}

encode = lambda s:[stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


In [744]:
data = torch.tensor(encode(text), dtype=torch.long)

# Split validation, and train data
n = int(0.9 * len(data))
train_data = data[:n] 
val_data = data[n:] 

In [745]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data[i:i+BLOCK_SIZE] for i in ix]) # 
    y = torch.stack([data[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x, y

In [746]:
xb, yb = get_batch('test')

print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(BATCH_SIZE): # batch dimension
    for t in range(BLOCK_SIZE): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[ 6,  1, 52, 53, 58,  1, 58, 47],
        [ 6,  1, 54, 50, 39, 52, 58, 43],
        [ 1, 58, 46, 47, 57,  1, 50, 47],
        [ 0, 32, 46, 43, 56, 43,  1, 42]])
targets:
torch.Size([4, 8])
tensor([[ 1, 52, 53, 58,  1, 58, 47, 50],
        [ 1, 54, 50, 39, 52, 58, 43, 58],
        [58, 46, 47, 57,  1, 50, 47, 60],
        [32, 46, 43, 56, 43,  1, 42, 53]])
----
when input is [6] the target: 1
when input is [6, 1] the target: 52
when input is [6, 1, 52] the target: 53
when input is [6, 1, 52, 53] the target: 58
when input is [6, 1, 52, 53, 58] the target: 1
when input is [6, 1, 52, 53, 58, 1] the target: 58
when input is [6, 1, 52, 53, 58, 1, 58] the target: 47
when input is [6, 1, 52, 53, 58, 1, 58, 47] the target: 50
when input is [6] the target: 1
when input is [6, 1] the target: 54
when input is [6, 1, 54] the target: 50
when input is [6, 1, 54, 50] the target: 39
when input is [6, 1, 54, 50, 39] the target: 52
when input is [6, 1, 54, 50, 39, 52] t

In [747]:
t = nn.Embedding(65, 32)
print(t.weight.shape)                       # (65, 32)
print(t(torch.zeros(4, 8, dtype=torch.long)).shape)   # (4, 8, 32)
a = torch.arange(8)
a

torch.Size([65, 32])
torch.Size([4, 8, 32])


tensor([0, 1, 2, 3, 4, 5, 6, 7])

In [748]:
class BigramLanguageModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, N_EMBEDDINGS)
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, N_EMBEDDINGS)
        self.llm_head = nn.Linear(N_EMBEDDINGS, vocab_size) # Linear Layer Construction: 32 numbers in, 16 out
    
    def forward(self, x, target=None):
        B, T = x.shape

        token_emedding = self.token_embedding_table(x) # Batch, Time, Embedding dimension, what i am 
        position_embedding = self.position_embedding_table(torch.arange(T)) # Time, Embedding dimension, where i am 
        x = token_emedding + position_embedding
        logit = self.llm_head(x) # batch, Time, Vocab size, passing x as input to the Linear layer

        if target == None:
            loss = None
        else:
            B, T, C = logit.shape
            logit = logit.view(B*T, C)
            target = target.view(B*T)
            loss = F.cross_entropy(logit, target)
        return logit, loss

    def generate(self, x, max_new_tokens):
        for _ in range(max_new_tokens):
            x_cond = x[:, -BLOCK_SIZE:]
            # Calling image.pngthe module runs forward.
            logit, _ = self(x_cond)
            # Only the last position's prediction is new
            logit = logit[:, -1, :] # (B, T, C) → (B, C)
            prob = F.softmax(logit, -1)
            sample = torch.multinomial(prob, num_samples=1)
            x = torch.cat((x, sample), dim=-1)
        return x

In [749]:
bi = BigramLanguageModel()    

x = encode("ROMEO:")
idx = torch.tensor([x])
out = bi.generate(idx, 200)[0].tolist()
decode(out)

"ROMEO:3KHvZvYMEEDZNyb!SZjyGFRX!DNZBK!NoUbDjYX!!ijT!hjmfHLWR3cyi.kvgvjWede$mZCNYZzMWv\nTDRxHFJxu?VHtVVBZmWE.jG'!!U!TCBZ'vIl3rZNjbhHYZjhmt!hw tAValfPjqYGgFyNqVbDJBR.\nOwZKoy!bDkFHy3sLfTmyvEZdZf.gJiPsMlZ?XYlGk$j"

In [750]:
# training using AdamW optimizer
optimizer = torch.optim.AdamW(bi.parameters(), lr=1e-3)

for i in range(TRAINING_STEPS):
    xb, yb = get_batch('train')
    logit, loss = bi(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss)


tensor(2.2290, grad_fn=<NllLossBackward0>)


In [751]:
x = encode("ROMEO:")
idx = torch.tensor([x])
out = bi.generate(idx, 200)[0].tolist()
decode(out)

'ROMEO: h cocet\nOM le thes:\nMdearo. te Are thy omatis tilfofour pin s llse aw-be f ENGL:\n\nMLOLUR!\nWhin, tave ve t gef ckithis anotowowougrnindago:\nALLAy by dowead t; ser.\nCINGo ind,\nPre pouie teave me ESer:\n'

Self attention basics
- We want to find a way for one token to talk with the other
- The simplest way to do that is for the current token to store the average of its vectors plus its prev vectors.
- **Affinity**: how much token in position i wants to hear from token in position j

x: t0 [1, 0]    t1 [2, 2]    t2 [0, 4]    t3 [4, 0]
xbow[0] = [1, 0]                       
xbow[1] = [(1+2)/2, (0+2)/2] = [1.5, 1]                 
xbow[2] = [(1+2+0)/3, (0+2+4)/3] = [1, 2]                            
xbow[3] = [1.75, 1.5]                 

there are multiple ways to do this, for loops, mat mul, and softmax

In [752]:
B, T, C = 4, 8, 32 
x  = torch.rand(B,T,C)

a = torch.ones((T,T))
tril = torch.tril(a)
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)
out = wei @ x
out.shape

torch.Size([4, 8, 32])

In [753]:
# self attention
B, T, C = 4, 8, 32 
x  = torch.rand(B,T,C)

HEAD_SIZE = 16

key = nn.Linear(C, HEAD_SIZE, bias=False)
query = nn.Linear(C, HEAD_SIZE, bias=False)
value = nn.Linear(C, HEAD_SIZE, bias=False)

k = key(x) # (4, 8, 32) @ (32, 16) -> (4, 8, 16)
q = query(x) # (4, 8, 32) @ (32, 16) -> (4, 8, 16)
v = value(x) # (4, 8, 32) @ (32, 16) -> (4, 8, 16)

kt = k.transpose(-1,-2) # (4, 16, 8)

wei = q @ kt
wei = wei * HEAD_SIZE ** 0.5

In [754]:
a = torch.ones(T,T)
tril = torch.tril(a)
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1)
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3363, 0.6637, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2663, 0.5977, 0.1360, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3043, 0.3959, 0.1251, 0.1747, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1728, 0.3763, 0.0996, 0.1528, 0.1986, 0.0000, 0.0000, 0.0000],
        [0.1165, 0.1981, 0.0496, 0.1165, 0.1349, 0.3843, 0.0000, 0.0000],
        [0.1077, 0.1570, 0.0546, 0.1146, 0.1541, 0.2201, 0.1919, 0.0000],
        [0.1343, 0.1834, 0.0448, 0.1007, 0.1376, 0.1654, 0.0692, 0.1647]],
       grad_fn=<SelectBackward0>)

In [755]:
out = wei @ v
out.shape

torch.Size([4, 8, 16])

In [756]:
class Head(nn.Module):
    def __init__(self, head_size) -> None:
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(N_EMBEDDINGS, head_size, bias=False)
        self.query = nn.Linear(N_EMBEDDINGS, head_size, bias=False)
        self.value = nn.Linear(N_EMBEDDINGS, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
    
    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        v = self.value(x)
        q = self.query(x)

        kt = k.transpose(-1, -2)

        wei = q @ kt
        wei = wei * self.head_size**-0.5 
        # Makes it so that tokens cant talk to future tokens
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        self.out = wei @ v
        return self.out
